# Lab 03 — Event Hubs Consumer Bronze

Consume live Wikimedia events from Azure Event Hubs with Spark Structured Streaming and store the raw payload plus Kafka/Event Hubs metadata in a Bronze Delta table.


## 1. Load the shared configuration


In [0]:
import socket

host = "evhpl24databricks.servicebus.windows.net"
port = 9093

try:
    with socket.create_connection(
        (host, port),
        timeout=10
    ):
        print(f"Connection successful: {host}:{port}")
except Exception as exc:
    print(f"Connection failed: {exc}")

In [0]:
%run ./lab03_config


## 2. Load the Event Hub secret and build Kafka settings

Azure Event Hubs exposes a Kafka-compatible endpoint. The namespace is extracted from the connection string without printing the secret.


In [0]:
import re

from pyspark.sql import functions as F


connection_string = dbutils.secrets.get(
    scope=secret_scope,
    key=eventhub_secret_key
)

endpoint_match = re.search(
    r"Endpoint=sb://([^/;]+)",
    connection_string
)

if endpoint_match is None:
    raise ValueError(
        "Could not extract the Event Hubs namespace "
        "from the connection string."
    )

eventhub_namespace_host = endpoint_match.group(1)

kafka_bootstrap_servers = (
    f"{eventhub_namespace_host}:9093"
)

escaped_connection_string = (
    connection_string
    .replace("\\", "\\\\")
    .replace('"', '\\"')
)

kafka_sasl_jaas_config = (
    "kafkashaded.org.apache.kafka.common.security.plain."
    "PlainLoginModule required "
    'username="$ConnectionString" '
    f'password="{escaped_connection_string}";'
)

print("Event Hubs Kafka configuration prepared.")
print(f"Bootstrap server: {kafka_bootstrap_servers}")
print(f"Event Hub topic: {eventhub_name}")
print(f"Consumer group: {eventhub_consumer_group}")


## 3. Create the streaming source

`startingOffsets = earliest` is used only when the checkpoint is new. After that, the checkpoint controls the next offset.


In [0]:
eventhub_stream_df = (
    spark.readStream
    .format("kafka")
    .option(
        "kafka.bootstrap.servers",
        kafka_bootstrap_servers
    )
    .option(
        "subscribe",
        eventhub_name
    )
    .option(
        "kafka.security.protocol",
        "SASL_SSL"
    )
    .option(
        "kafka.sasl.mechanism",
        "PLAIN"
    )
    .option(
        "kafka.sasl.jaas.config",
        kafka_sasl_jaas_config
    )
    .option(
        "startingOffsets",
        "earliest"
    )
    .option(
        "failOnDataLoss",
        "false"
    )
    .load()
)

print("Event Hubs streaming source created.")


## 4. Prepare the Bronze schema

Bronze keeps the original JSON event unchanged and adds transport and ingestion metadata.


In [0]:
bronze_df = (
    eventhub_stream_df
    .select(
        F.col("value").cast("string").alias("raw_event"),
        F.col("key").cast("string").alias("message_key"),
        F.col("topic").alias("eventhub_name"),
        F.col("partition").alias("eventhub_partition"),
        F.col("offset").alias("eventhub_offset"),
        F.col("timestamp").alias("eventhub_enqueued_at"),
        F.col("timestampType").alias("timestamp_type"),
        F.current_timestamp().alias("bronze_ingested_at"),
        F.lit(producer_id).alias("expected_producer_id")
    )
)

print("Bronze streaming DataFrame prepared.")
bronze_df.printSchema()


## 5. Write raw events to the Bronze Delta table

The query runs continuously until stopped manually. The checkpoint guarantees incremental processing and restart recovery.


In [0]:
bronze_query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        eventhub_bronze_checkpoint_path
    )
    .queryName("lab03_eventhub_bronze_ingestion")
    .trigger(availableNow=True)
    .toTable(eventhub_bronze_table)
)

print("Bronze availableNow stream started.")
print(f"Target table: {eventhub_bronze_table}")
print(f"Checkpoint: {eventhub_bronze_checkpoint_path}")
print(f"Query ID: {bronze_query.id}")

# Wait until all currently available Event Hub events are processed.
bronze_query.awaitTermination()

print("Bronze ingestion completed.")

if bronze_query.lastProgress:
    print(
        "Input rows processed:",
        bronze_query.lastProgress.get("numInputRows", 0)
    )


## 6. Monitor the Bronze stream

Run this cell while the query is active.


In [0]:
bronze_query.status


In [0]:
bronze_query.lastProgress


## 7. Stop the stream when enough events have arrived

Run this cell manually when you want to stop ingestion.


In [0]:
if bronze_query.isActive:
    bronze_query.stop()
    bronze_query.awaitTermination()

print("Bronze stream stopped.")


## 8. Verify the Bronze table


In [0]:
bronze_count = spark.table(
    eventhub_bronze_table
).count()

print(f"Bronze row count: {bronze_count}")

display(
    spark.table(eventhub_bronze_table)
    .orderBy(F.col("bronze_ingested_at").desc())
    .limit(20)
)
